# 05b - (Tahap 1) Eksplorasi dataset

In [1]:
import json, re, os
from collections import Counter, defaultdict
import random

## 1. Konfigurasi

In [2]:
INPUT_PATH = "/kaggle/input/notebooks/muhammadnabilul/05-generating-qa/output_05a/raw_records.jsonl"
OUTPUT_DIR = "output_05b_exploration"
os.makedirs(OUTPUT_DIR, exist_ok=True)
 
with open(INPUT_PATH, encoding="utf-8") as f:
    records = [json.loads(l) for l in f if l.strip()]
 
print(f"Total records: {len(records)}\n")
rng = random.Random(42)

Total records: 24426



## 2. Distribusi

In [3]:
print("\n[Task Type]")
for k, v in Counter(r["task_type"] for r in records).most_common():
    print(f"  {k:<30}: {v:>6} ({v/len(records)*100:.1f}%)")
 
print("\n[Subtask Type]")
for k, v in Counter(r["subtask_type"] for r in records).most_common():
    print(f"  {k:<35}: {v:>6} ({v/len(records)*100:.1f}%)")
 
print("\n[Reasoning Source]")
for k, v in Counter(r.get("reasoning_source", "?") for r in records).most_common():
    print(f"  {k:<25}: {v:>6} ({v/len(records)*100:.1f}%)")
 
print("\n[Generation Focus]")
for k, v in Counter(r.get("generation_focus", "?") for r in records).most_common():
    print(f"  {k:<35}: {v:>6} ({v/len(records)*100:.1f}%)")
 
print("\n[Parse Status]")
for k, v in Counter(r.get("parse_status", "?") for r in records).most_common():
    print(f"  {k:<20}: {v:>6} ({v/len(records)*100:.1f}%)")


[Task Type]
  qa                            :  22635 (92.7%)
  summary                       :   1791 (7.3%)

[Subtask Type]
  factual_retrieval                  :   3137 (12.8%)
  definition_clarification           :   2883 (11.8%)
  trend_interpretation               :   2840 (11.6%)
  comparative_analysis               :   2828 (11.6%)
  multi_source_synthesis             :   2825 (11.6%)
  causal_reasoning                   :   2822 (11.6%)
  methodology_explanation            :   2775 (11.4%)
  source_navigation                  :   2525 (10.3%)
  document_overview                  :   1791 (7.3%)

[Reasoning Source]
  single_paragraph         :  15582 (63.8%)
  multi_paragraph          :   5696 (23.3%)
  whole_section            :   3147 (12.9%)
  multiple_paragraph       :      1 (0.0%)

[Generation Focus]
  literal_reading                    :   5662 (23.2%)
  evidence_integration               :   5662 (23.2%)
  concept_understanding              :   5658 (23.2%)
  analytical

## 3. Panjang intruction & Response

In [4]:
inst_lens = [len(r["instruction"].split()) for r in records]
resp_lens = [len(r["response"].split()) for r in records]
 
def stats(vals, label):
    import statistics
    print(f"\n[{label}]")
    print(f"  min={min(vals)} max={max(vals)} "
          f"mean={statistics.mean(vals):.1f} median={statistics.median(vals):.0f}")
    brackets = [(0,5),(5,10),(10,20),(20,50),(50,100),(100,200),(200,9999)]
    for lo, hi in brackets:
        n = sum(1 for v in vals if lo <= v < hi)
        if n: print(f"  [{lo:>4}–{hi:>4} kata): {n:>6} ({n/len(vals)*100:.1f}%)")
 
stats(inst_lens, "Instruction (kata)")
stats(resp_lens, "Response (kata)")
 
very_short_inst = [r for r in records if len(r["instruction"].split()) < 5]
very_short_resp = [r for r in records if len(r["response"].split()) < 6]
print(f"\nInstruction < 5 kata  : {len(very_short_inst)}")
print(f"Response < 6 kata     : {len(very_short_resp)}")
if very_short_resp:
    print("\nSample response sangat pendek:")
    for r in rng.sample(very_short_resp, min(3, len(very_short_resp))):
        print(f"  [{r['subtask_type']}] Q: {r['instruction'][:80]}")
        print(f"  A: {r['response']}")


[Instruction (kata)]
  min=4 max=77 mean=16.9 median=16
  [   0–   5 kata):      8 (0.0%)
  [   5–  10 kata):   1674 (6.9%)
  [  10–  20 kata):  15726 (64.4%)
  [  20–  50 kata):   6997 (28.6%)
  [  50– 100 kata):     21 (0.1%)

[Response (kata)]
  min=4 max=340 mean=40.3 median=32
  [   0–   5 kata):      1 (0.0%)
  [   5–  10 kata):    703 (2.9%)
  [  10–  20 kata):   4431 (18.1%)
  [  20–  50 kata):  13298 (54.4%)
  [  50– 100 kata):   4667 (19.1%)
  [ 100– 200 kata):   1287 (5.3%)
  [ 200–9999 kata):     39 (0.2%)

Instruction < 5 kata  : 8
Response < 6 kata     : 20

Sample response sangat pendek:
  [factual_retrieval] Q: Berapa persentase IPM DI Yogyakarta?
  A: IPM DI Yogyakarta sebesar 94,13.
  [factual_retrieval] Q: Berapa IPM Indonesia pada tahun 2022?
  A: IPM 2022 tercatat sebesar 73,81.
  [factual_retrieval] Q: Tabel nomor berapa yang menampilkan persentase penduduk berumur 0-6 tahun yang p
  A: Tabel tersebut adalah Tabel 3.9.1.


## 4. Pattern sumber

In [5]:
meta_patterns = {
    "awalan_teks_sumber":   r"^(berdasarkan|menurut|sesuai|dalam)\s+teks sumber",
    "awalan_konteks":       r"^(berdasarkan|menurut|sesuai|dalam)\s+konteks",
    "awalan_passages":      r"^(berdasarkan|menurut|sesuai|dalam)\s+passages",
    "awalan_dokumen_ini":   r"^(berdasarkan|menurut|sesuai|dalam)\s+dokumen ini",
    "sebut_teks_sumber_ini":r"teks sumber ini",
    "sebut_passages":       r"\bpassages\b",
    "sebut_bagian_ini":     r"bagian ini",
    "sebut_di_atas":        r"\bdi atas\b",
    "sebut_konteks_ini":    r"konteks ini",
}
 
meta_found = defaultdict(list)
for r in records:
    text = f"{r['instruction']} {r['response']}".lower()
    for name, pat in meta_patterns.items():
        if re.search(pat, text, re.IGNORECASE):
            meta_found[name].append(r)
 
for name, recs in sorted(meta_found.items(), key=lambda x: -len(x[1])):
    print(f"\n[{name}]: {len(recs)} records ({len(recs)/len(records)*100:.1f}%)")
    sample = rng.sample(recs, min(2, len(recs)))
    for r in sample:
        print(f"  [{r['subtask_type']}]")
        print(f"  Q: {r['instruction'][:120]}")
        print(f"  A: {r['response'][:200]}")


[sebut_di_atas]: 372 records (1.5%)
  [causal_reasoning]
  Q: Mengapa data persentase anak usia dini yang memiliki akses terhadap fasilitas cuci tangan untuk laki-laki di DKI Jakarta
  A: Data untuk laki-laki di DKI Jakarta pada Tabel 4.35 memiliki Relative Standard Error (RSE) sebesar 34,25%. RSE ini berada di atas 25% tetapi tidak melebihi 50%. Menurut catatan tabel, jika RSE >25% te
  [document_overview]
  Q: Uraikan secara komprehensif kriteria kelayakan hunian dan metode evaluasi kualitas data yang digunakan dalam publikasi i
  A: Kelayakan hunian mencakup empat aspek: atap (beton, genteng, seng, kayu/sirap), dinding (tembok, plesteran anyaman bambu/kawat, kayu/papan, batang kayu), lantai (marmer/granit, keramik, parket/vinil/k

[sebut_konteks_ini]: 12 records (0.0%)
  [definition_clarification]
  Q: Apa yang dimaksud dengan 'politik uang' dalam konteks temuan survei ini?
  A: Politik uang dalam konteks ini merujuk pada tawaran uang, barang, atau fasilitas kepada masyarakat untuk

## 5. Angka tidak terverifikasi

In [6]:
NUM_ID = re.compile(r"\b\d{1,3}(?:\.\d{3})*(?:,\d+)?\b|\b\d+,\d+\b|\b\d{2,}\b")
 
def check_numbers(r):
    nums = NUM_ID.findall(r["response"])
    if not nums:
        return [], []
    ctx = r.get("context", "")
    unmatched = [n for n in nums if n not in ctx]
    return nums, unmatched
 
no_match_count = 0
sample_unmatched = []
for r in records:
    nums, unmatched = check_numbers(r)
    if unmatched:
        no_match_count += 1
        if len(sample_unmatched) < 5:
            sample_unmatched.append((r, nums, unmatched))
 
print(f"\nRecord dengan angka tak cocok ke context: {no_match_count} ({no_match_count/len(records)*100:.1f}%)")
print("\nSample (5 contoh):")
for r, nums, unmatched in sample_unmatched:
    print(f"\n  [{r['subtask_type']}]")
    print(f"  Q: {r['instruction'][:100]}")
    print(f"  A: {r['response'][:200]}")
    print(f"  Semua angka di response: {nums[:10]}")
    print(f"  Tidak cocok di context : {unmatched[:5]}")


Record dengan angka tak cocok ke context: 1237 (5.1%)

Sample (5 contoh):

  [methodology_explanation]
  Q: Bagaimana metode pengukuran Indeks Perilaku Anti Korupsi (IPAK) di Indonesia?
  A: IPAK diukur melalui Survei Perilaku Anti Korupsi (SPAK) yang dilakukan oleh BPS, dengan menghitung persentase penduduk yang tidak melakukan perilaku koruptif dalam 12 bulan terakhir pada beberapa dime
  Semua angka di response: ['12', '0', '100']
  Tidak cocok di context : ['100']

  [methodology_explanation]
  Q: Bagaimana cara menghitung skor Corruption Perceptions Index (CPI) Indonesia?
  A: Skor CPI Indonesia dihitung oleh Transparency International dengan menggabungkan 13 survei dan penilaian dari berbagai lembaga internasional yang mengukur persepsi tingkat korupsi di sektor publik. Sk
  Semua angka di response: ['13', '0', '100', '0', '100']
  Tidak cocok di context : ['100', '100']

  [methodology_explanation]
  Q: Bagaimana cara Transparency International mengukur Indeks Persepsi Korupsi 

## 6. Catatan proses berpikir dari model

In [7]:
thinking_patterns = {
    "tidak_disebut_eksplisit": r"\(tidak disebut eksplisit",
    "dihitung_dari":           r"\(dihitung dari",
    "estimasi":                r"\(estimasi\)",
    "kemungkinan":             r"\(kemungkinan\)",
    "diasumsikan":             r"\(diasumsikan",
    "perkiraan":               r"\bperkiraan\)",
}
 
thinking_found = defaultdict(list)
for r in records:
    resp = r["response"].lower()
    for name, pat in thinking_patterns.items():
        if re.search(pat, resp, re.IGNORECASE):
            thinking_found[name].append(r)
 
for name, recs in sorted(thinking_found.items(), key=lambda x: -len(x[1])):
    print(f"\n[{name}]: {len(recs)} records")
    for r in rng.sample(recs, min(2, len(recs))):
        print(f"  [{r['subtask_type']}] A: {r['response'][:250]}")

## 7. Duplikat Intruksi/pertanyaan

In [8]:
norm_inst = Counter(
    re.sub(r"[^\w\s]", "", r["instruction"].lower()).strip()
    for r in records
)
exact_dups = {k: v for k, v in norm_inst.items() if v > 1}
print(f"\nInstruction exact duplicate: {len(exact_dups)} cluster, "
      f"{sum(exact_dups.values())} records total")
 
print("\nTop 5 instruction yang paling sering muncul:")
for inst, count in sorted(exact_dups.items(), key=lambda x: -x[1])[:5]:
    print(f"  ({count}x) {inst[:100]}")


Instruction exact duplicate: 321 cluster, 947 records total

Top 5 instruction yang paling sering muncul:
  (35x) apa yang dimaksud dengan sampling error dalam tabel ini
  (25x) apa yang dimaksud dengan relative standard error rse dalam tabel ini
  (23x) apa yang dimaksud dengan sampling error dalam konteks tabel ini
  (22x) apa yang dimaksud dengan selang kepercayaan 95 dalam tabel ini
  (21x) apa yang dimaksud dengan selang kepercayaan 95 dalam konteks tabel ini


## 8. Pertanyaan mirip jawaban

In [9]:
import difflib
high_sim = []
for r in records:
    instr = r["instruction"].lower()
    resp  = r["response"].lower()
    sim = difflib.SequenceMatcher(None, instr, resp).ratio()
    r["_sim"] = sim
    if sim > 0.85:
        high_sim.append(r)
 
print(f"\nSimilarity > 0.85: {len(high_sim)} records ({len(high_sim)/len(records)*100:.1f}%)")
print("\nSample (perhatikan apakah ini legitimate untuk definition_clarification):")
by_subtask = defaultdict(list)
for r in high_sim:
    by_subtask[r["subtask_type"]].append(r)
for subtask, recs in sorted(by_subtask.items(), key=lambda x: -len(x[1]))[:4]:
    print(f"\n  [{subtask}] — {len(recs)} records")
    r = recs[0]
    print(f"  Q: {r['instruction'][:120]}")
    print(f"  A: {r['response'][:200]}")
    print(f"  sim={r['_sim']:.3f}")


Similarity > 0.85: 1021 records (4.2%)

Sample (perhatikan apakah ini legitimate untuk definition_clarification):

  [factual_retrieval] — 968 records
  Q: Berapa persentase pelaku usaha yang mengalami pembayaran melebihi ketentuan pada tahun 2024?
  A: Persentase pelaku usaha yang mengalami pembayaran melebihi ketentuan pada tahun 2024 adalah 18,81 persen.
  sim=0.853

  [source_navigation] — 38 records
  Q: Di gambar nomor berapa persentase pendapat masyarakat tentang seseorang menggunakan barang milik anggota keluarga tanpa 
  A: Persentase pendapat masyarakat tentang seseorang menggunakan barang milik anggota keluarga tanpa seizin pemiliknya ditampilkan pada Gambar 2.5.
  sim=0.860

  [trend_interpretation] — 6 records
  Q: Bagaimana tren proporsi anak umur di bawah 5 tahun yang kelahirannya dicatat oleh lembaga pencatatan sipil dari tahun 20
  A: Proporsi anak umur di bawah 5 tahun yang kelahirannya dicatat oleh lembaga pencatatan sipil menunjukkan peningkatan dari tahun 2022 hin

## 9. Validitas pertanyaan/intruksi

In [10]:
ALLOWED_TASKS = {"qa", "summary"}
ALLOWED_SUBTASKS = {
    "factual_retrieval", "source_navigation",
    "definition_clarification", "methodology_explanation",
    "trend_interpretation", "causal_reasoning",
    "comparative_analysis", "multi_source_synthesis",
    "document_overview",
}
invalid_task    = [r for r in records if r.get("task_type") not in ALLOWED_TASKS]
invalid_subtask = [r for r in records if r.get("subtask_type") not in ALLOWED_SUBTASKS]
missing_fields  = [r for r in records if not r.get("instruction") or not r.get("response")
                   or not r.get("context") or not r.get("context_chunk_ids")]
 
print(f"\nInvalid task_type    : {len(invalid_task)}")
print(f"Invalid subtask_type : {len(invalid_subtask)}")
print(f"Missing required field: {len(missing_fields)}")
 
if invalid_subtask:
    print("\nSubtask tidak dikenali:")
    for k, v in Counter(r["subtask_type"] for r in invalid_subtask).most_common():
        print(f"  {k}: {v}")


Invalid task_type    : 0
Invalid subtask_type : 0
Missing required field: 0


## 10. Ringkasan

In [11]:
total = len(records)
print(f"\nTotal records: {total}")
print(f"\n[Calon HARD GATE — buang]")
print(f"  schema invalid (task/subtask)       : {len(invalid_task)+len(invalid_subtask)}")
print(f"  missing required fields             : {len(missing_fields)}")
print(f"  response < 6 kata                   : {len(very_short_resp)}")
thinking_total = len(set(id(r) for recs in thinking_found.values() for r in recs))
print(f"  thinking artifacts (estimasi, dll)  : {thinking_total}")
 
print(f"\n[Calon CLEANING — strip, bukan buang]")
meta_total = len(set(id(r) for recs in meta_found.values() for r in recs))
print(f"  meta-reference awalan               : {meta_total} ({meta_total/total*100:.1f}%)")
 
print(f"\n[Calon SOFT SCORE — penalti tapi tidak buang]")
print(f"  angka tidak terverifikasi           : {no_match_count} ({no_match_count/total*100:.1f}%)")
print(f"  high similarity inst≈resp           : {len(high_sim)} ({len(high_sim)/total*100:.1f}%)")
 
print(f"\n[Calon NEAR-DUPLICATE — perlu embedding check]")
print(f"  exact duplicate instruction         : {sum(exact_dups.values())} (minimum, sebelum semantic dedup)")
 
print(f"\n[Export exploration ke {OUTPUT_DIR}/]")
with open(f"{OUTPUT_DIR}/meta_reference_samples.jsonl", "w") as f:
    for recs in meta_found.values():
        for r in recs:
            f.write(json.dumps({"record_id": r["record_id"], "subtask": r["subtask_type"],
                                "instruction": r["instruction"], "response": r["response"]},
                               ensure_ascii=False) + "\n")
 
with open(f"{OUTPUT_DIR}/thinking_artifact_samples.jsonl", "w") as f:
    for recs in thinking_found.values():
        for r in recs:
            f.write(json.dumps({"record_id": r["record_id"], "subtask": r["subtask_type"],
                                "response": r["response"]}, ensure_ascii=False) + "\n")
 
print("Done.")


Total records: 24426

[Calon HARD GATE — buang]
  schema invalid (task/subtask)       : 0
  missing required fields             : 0
  response < 6 kata                   : 20
  thinking artifacts (estimasi, dll)  : 0

[Calon CLEANING — strip, bukan buang]
  meta-reference awalan               : 398 (1.6%)

[Calon SOFT SCORE — penalti tapi tidak buang]
  angka tidak terverifikasi           : 1237 (5.1%)
  high similarity inst≈resp           : 1021 (4.2%)

[Calon NEAR-DUPLICATE — perlu embedding check]
  exact duplicate instruction         : 947 (minimum, sebelum semantic dedup)

[Export exploration ke output_05b_exploration/]
Done.
